# Partial Differential Equations I

Chapter 10 of the TN23015 book: its explanation and its exercises. Run the first cell before anything else, then work down the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors
from time import time
from tn23015 import check_answers, given, load_data, md, print, row, show, slider

In this lecture, we will explore numerically solving partial differential equations (PDEs) that have the form of boundary value problems using various techniques of relaxation.

> **Learning goals**
>
> 1. Implement the Jacobi and Gauss-Seidel methods of solving PDEs 
> 2. Speed up calculation by implementing successive over-relaxation (SOR)
> 3. Solve for the electrostatic potential $\phi$ for 2-dimensional (2D) problems using SOR including fixed charges
> 4. Calculate the charge density, including the charge induced on the boundaries, from a simulation of the 2D Poisson equation

This will be a handy plotting function that we'll use a few times later.

In [ ]:
def visualise(array, normalisation=0.5, line_cut_y=80, autoscale=False):
    fig, axes = plt.subplots(1, 2, figsize=(14,6))

    lims = [0, len(array[0])-1] # Assumes square matrix
    zlims = [np.min(array), np.max(array)]
    assert line_cut_y < lims[-1], "The line_cut_y that you ask for should be less than the size of the array"
    assert isinstance(line_cut_y, int), "The line_cut_y that you ask for should be a whole number (integer)"

    norm = colors.PowerNorm(gamma=normalisation, vmin=np.min(array), vmax=np.max(array))
    im = axes[0].imshow(array, cmap='RdBu_r', origin='lower',aspect='auto', norm=norm)
    axes[0].hlines(y=line_cut_y, xmin=0, xmax=len(array[0]), color='black', linestyle='--')
    plt.colorbar(im, ax=axes[0], cmap='RbBu_r', label='z')

    axes[0].set_ylim(*lims)
    axes[0].set_xlim(*lims)
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('y')

    axes[1].plot(array[line_cut_y], color='black', linestyle='--')
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('z')
    if autoscale:
        axes[1].relim()
        axes[1].autoscale_view()
    else:
        axes[1].set_ylim(*zlims)
        axes[1].autoscale_view()
    return fig

show()

## Poisson's equation in 2D

We will look specifically in this course at the solutions to a specific, linear partial differential equation known as Poisson's equation:

$$
\nabla^2 \phi = 
\frac{\partial^2\phi}{\partial x^2} +
\frac{\partial^2\phi}{\partial y^2} + 
\frac{\partial^2\phi}{\partial z^2} =
-\rho(x,y,z)
$$

This equation is hopefully well known to you from your course in [electrostatics](https://en.wikipedia.org/wiki/Electrostatics): it is the equation that determines the electrostatic potential $\phi(\vec{r})$ for a given distribution of charges $\rho(\vec{r})$. 

In this lecture, we will discuss how to solve a two-dimensional version of the equation.
By working in 2D, we will keep the calculation times relatively short and it will be easy to visualize our results.
The technique, however, can easily be extended into 3D in an obvious way.

What does this mean physically?
Of course, Maxwell's equations apply to problems in 3-dimensions: we can't just cut one of the dimensions off!
However, if our system is translationally invariant in one dimension (say, the $z$ direction for example), then we know that $\phi$ cannot depend on $z$ since if we move in the $z$ direction, everything looks the same.
In this case, the derivative in the $z$ direction will be zero:

$$
\frac{\partial^2\phi}{\partial z^2} = 0
$$

leading us to a two-dimensional version of the Poisson equation:

$$
\frac{\partial^2\phi}{\partial x^2} +
\frac{\partial^2\phi}{\partial y^2} =
-\rho(x,y)
$$

What would something like this look like physically?
A good example is the two wires that make up a simple transmission line:

![](https://raw.githubusercontent.com/curiousbeams/lecture-tn23015-computational-science/main/figures/transmission_line.png)

If we approximate the wires as infinitely long, or are interested only in places near the wires and also far from their ends, the 2D Poisson equation above will give us a good approximation of the electrostatic potential in the plane perpendicular to the wire direction. 

We will start with the simple case where $\rho(x,y) = 0$, leading to Laplace's equation:

$$
\frac{\partial^2\phi}{\partial x^2} +
\frac{\partial^2\phi}{\partial y^2} = 0
$$

We will use this to explore various techniques for solving it using finite differences and different relaxation techniques, then move on to solving physical problems in electrostatics. 

## Finite Difference method

The first step in solving the problem numerically is to map the continuous $x,y$ plane into discrete points.
Here, we will consider discrete points with a spacing $a$. 

The next step is to approximate the partial derivatives with finite differences.
Doing so will lead to:

$$
\frac{\partial^2\phi}{\partial x^2} +
\frac{\partial^2\phi}{\partial y^2} \approx 
\frac{
\phi(x+a,y) + 
\phi(x-a,y) + 
\phi(x,y+a) + 
\phi(x,y-a) -
4 \phi(x,y)}
{a^2}
$$

All we need to do now is to equate this to zero and find the solutions of the resulting coupled (linear) equations.
How many equations will we have?
If we choose a grid of 100x100 points in our $x,y$ plane, we will have 10,000 equations!
That's a lot!
Coding this into a matrix is quite daunting...

### Jacobi method for solving the linear equations

Fortunately, there is a simpler method for solving these 10,000 equations, which is known as "relaxation". 

The problem we are looking at is a boundary-value problem: to be able to solve for the potential on our grid, we need to know the value of the potential on the boundaries of our simulation.
This is our "known" starting point, and we need to then find the solutions in the middle that solve the Laplace (or the Poisson) equation. 

The way relaxation works is the following: we start with our fixed boundary condition on the outside of our simulation "box".
We then start with an initial guess at the potential everywhere inside the box.
What do we choose for this?
It turns out that it **doesn't matter** what we choose: the techniques we will look at will always work no matter what you chose (which is handy!).

(Note that here, we will consider only "metallic" fixed-potential boundary conditions, also known as the Dirichlet condition.
One alternative is also the Neumann boundary condition, specifying no electric field perpendicular to the boundary.
Another is periodic boundary conditions.
In addition, there are more [exotic boundary conditions](https://arxiv.org/abs/1510.04400) that allow, among other things, the simulation of an open boundary to infinity.)

Once we have our boundary set and our initial guess, we then iterate through all the points in our simulation and replace the value of the potential at iteration $N+1$ with the value that would solve the equation we are solving, given the values of its neighbors at that step of the iteration $N$. With $\rho(x,y) = 0$, this becomes:

$$
\phi_{N+1}(x,y) = 
\frac{\phi_N(x+a,y) + 
\phi_N(x-a,y) + 
\phi_N(x,y+a) + 
\phi_N(x,y-a)}{4}
$$

Note that for $\rho = 0$, the lattice spacing $a$ falls out: we don't even need to know the spacing to do the calculation, just the aspect ratio.

Miraculously (?), if we just keep doing this for a while, the potential will magically converge to the correct solution!

How do we know when to stop?
One way is to keep track of the largest change in $\phi$ of any of the pixels in our simulation.
If this change in potential goes below a target accuracy that we configure, then we can stop iterating our simulation.

### Exercise 10.1

Consider a square box with the left, right, and bottom sides grounded, and the top side set to a voltage of 1V.
In our finite difference technique, we will divide the two-dimensional $x-y$ space into an $M \times M$ grid, illustrated here for $M = 11$: 

![](https://raw.githubusercontent.com/curiousbeams/lecture-tn23015-computational-science/main/figures/box.png)

The outer boundary points are fixed by the boundary condition, and are not updated when we are solving for $\phi$: instead they remain at a fixed potential.
We need to keep track of this so that we do not accidentally update any points that should correspond to a fixed external potential. 

Note that our update equation above does not depend on the spacing between our grid points: the Laplace equation has the same solution regardless of whether the box is 1 meter or 1 nm across!
For the Laplace equation, we do not have to keep track of what the spacing between our points is.

Your task is to fill in the code below to use Jacobi's method to calculate $\phi(x,y)$.

During your calculation, keep track of the absolute value of the largest change `delta_max` in $\phi$ during that iteration in a list so that we can plot later the convergence as a function of the number of iterations.
Use a threshold of 0.1 mV for $\phi$.

To benchmark the computational speed of our simulation, we will keep track of the running time and the code speed.

In [ ]:
target_accuracy = 1e-4

# The size of the simulation grid
M = 101
phi = np.zeros([M,M])
phi[-1,1:-1] = 1

# A second matrix for storing the values calculated for the next
# iteration
phinew = phi.copy()

# A matrix to keep track of which points in our simulation 
# are fixed voltages. In this case, it is only the boundaries
# of the simulation
fixed = np.empty([M,M]) 
fixed[:,:] = False # interior points are not fixed, they should be iterated
fixed[0,:] = True
fixed[-1,:] = True
fixed[:,0] = True
fixed[:,-1] = True

# Keep track of the biggest update. 
# When this is smaller than our target accuracy, 
# we will stop iterating
delta_max = 1
delta_max_list = []

t1 = time()

# Set `unfinished` to False once you have written the code below, so that it can run.
unfinished = True

t2 = None
if not unfinished:
    while delta_max > target_accuracy:
        for _i in range(M):
            for _j in range(M):
                if fixed[_i,_j]:
                    ...

                    #phinew[_i,_j] = ...
                else:
                    ...

                    #phinew[_i,_j] = ....
        #delta_max =....
        delta_max_list.append(delta_max)
        print("N_iter %d delta_max %e\r" % (len(delta_max_list), delta_max), end='')

        # Now that we're done, phi becomes phinew. A sneaky trick: if we swap the two, 
        # we don't need to copy the whole array! 
        phi,phinew = phinew,phi
    t2 = time()

    print("\nTotal running time: %.2f min" % ((t2-t1)/60))
    print("Code speed: %.1f iterations per second" %(len(delta_max_list)/(t2-t1)))

# Do not edit or remove the boiler-plate code below.
show()

In [ ]:
check_answers(phi=phi, delta_max_list=delta_max_list, key="answer_13_1a", when=not unfinished)

To explore your data, you can use the plotting function from the top of the page to make a handy "data explorer" that can show you both color scale plots and also line cuts. 

Now let's use this function to explore the $\phi$ that you calculated:

In [ ]:
# Do not edit or remove the boiler-plate code below.
fig = None
if given(phi):
    fig = visualise(phi, normalisation=0.5, line_cut_y=80)
    fig.show()

show(phi=phi)

### Exercise 10.2

Make a plot of `delta_max` as a function of the iteration number.
The y-axis should be a log scale.
As always, your plot should have appropriate labels with units where applicable.

In [ ]:
# Your code here:

# Do not edit or remove the boiler-plate code below.
show()

## Gauss-Seidel method

A speedup of the Jacobi method can be obtained by using the _already updated_ calculated values of $\phi$ instead of waiting for the next round. 

For doing this, we have to have a different way of keeping track of `delta_max` than we used above, and so the code will look a bit different.
A handy (and computationally efficient) way of doing this in python is to keep track of all the `delta` in a matrix and then use `np.max(np.abs(delta))` to find the largest one afterwards.

(You might think: we could add an `if` statement in our loop to keep track of `delta_max`: why do we need a full matrix?
This is what we initially thought too but it turns out that this slows your code down a lot and you're much faster just keeping the full matrix and using `np.max()` at the end.
Why?
Python is an interpreted language, which means adding anything to a hand-coded loop is just a really, really bad idea if you don't have to.
Because `np.max()` is vectorised: it runs pre-compiled, not interpreted, loops, which can be hundreds of times faster than a `for` loop written in Python.)

### Exercise 10.3

Perform the calculation of Exercise 10.1 with the Gauss-Seidel method.

In [ ]:
target_accuracy_2 = 1e-4

M_2 = 101
phi_2 = np.zeros([M_2,M_2])
delta = phi_2.copy()
phi_2[-1,1:-1] = 1

# A matrix to keep track of which points in our simulation 
# are fixed voltages. In this case, it is only the boundaries
# of the simulation
fixed_2 = np.empty([M_2,M_2])
fixed_2[:,:] = False
fixed_2[0,:] = True
fixed_2[-1,:] = True
fixed_2[:,0] = True
fixed_2[:,-1] = True

# Keep track of the biggest update. 
# When this is smaller than our target accuracy, 
# we will stop iterating
delta_max_2 = 1
delta_max_list_2 = []

t1_2 = time()

# Set `unfinished_2` to False once you have written the code below, so that it can run.
unfinished_2 = True

t2_2 = None
if not unfinished_2:
    while delta_max_2 > target_accuracy_2:
        for _i in range(M_2):
            for _j in range(M_2):
                if not fixed_2[_i,_j]:
                    ...

                    #delta[_i,_j] = ...
        delta_max_2 = np.max(np.abs(delta))
        delta_max_list_2.append(delta_max_2)
        print("N_iter %d delta_max %e\r" % (len(delta_max_list_2), delta_max_2), end='')   
    t2_2 = time()
    print("\nTotal running time: %.2f min" % ((t2_2-t1_2)/60))
    print("Code speed: %.1f iterations per second" %(len(delta_max_list_2)/(t2_2-t1_2)))

# Do not edit or remove the boiler-plate code below.
show()

In [ ]:
check_answers(phi_2=phi_2, delta_max_list_2=delta_max_list_2, key="answer_13_2a", when=not unfinished_2)

You can check that it looks the same:

In [ ]:
# Do not edit or remove the boiler-plate code below.
fig_2 = None
if given(phi_2):

    # Notebook code
    fig_2 = visualise(phi_2, normalisation=0.5, line_cut_y=80)
    fig_2.show()

show(phi_2=phi_2)

Although the code is not that much faster in total time (because of the slightly more complicated stuff going on in the `for` loops), it converges in fewer iterations.

### Exercise 10.4

Plot `delta_max` vs iteration number for both the Jacobi method and the Gauss-Seidel method.
Use a logscale for the y-axis.

In [ ]:
# Your code here:

# Do not edit or remove the boiler-plate code below.
show()

## Successive over-relaxation

Although we did not gain much speed with the Gauss-Seidel technique, there is a small trick we can play with it to make it MASSIVELY faster.

The trick is as follows: instead of correcting `phi` to the value that would solve the equation based on its neighboring values, we go *a bit* further: 

$$
\phi_{new}(x,y) = \phi_{old}(x,y) + \omega \delta
$$

where $\delta$ is the correction that you would have calculated in the Gauss-Seidel method and $\omega$ is a number between 0 and 2. With $\omega > 1$, this technique is known as successive over-relaxation (SOR).
It turns out, if you pick the right $\omega$, your simulation can run MUCH MUCH faster. 

Picking the optimal value of $\omega$ is not easy, and depends in detail on the size and even configuration of your simulation.
The Gauss-Seidel method corresponds to $\omega = 1$.
For $\omega > 2$, the method does not converge (it is unstable).
And, for nonlinear Poisson equations (in which $\rho$ depends on $\phi$ in a nonlinear way, which Gary Steele studied in his PhD), under-relaxing with $\omega < 1$ can help unstable simulations converge.
In general, for regular linear PDEs and large simulations, a value close to 2 is a good guess, and guessing a bit high is better than guessing too low.
But even if you guess a bit wrong, it still can speed up your simulation very significantly.

(In particular, it can change the scaling with simulation size from $N^2$ to $N^{1.5}$, which is a huge deal!
You can read chapter 5 of [Gary Steele's PhD thesis](http://nsweb.tn.tudelft.nl/~gsteele/thesis/Gary_Steele.compressed_images.pdf) if you would like to learn some more about the speed and applications of relaxation technique.)

As you will see, even for these small simulation sizes, it can significantly speed up the calculation.

### Exercise 10.5

Modify your code from Exercise 10.3 to implement SOR.
Use an SOR parameter of 1.95, which works well by trial-and-error.

In [ ]:
# Fill these in as you work through the exercise.
delta_max_list_3 = phi_3 = None

SOR_parameter = 1.95
target_accuracy_3 = 1e-4
t1_3 = time()

# phi_3 = ...
# delta_max_list_3 = ...

# Do not edit or remove the boiler-plate code below.
if given(delta_max_list_3, phi_3):
    _t2 = time()
    print("\nTotal running time: %.2f min" % ((_t2-t1_3)/60))
    print("Code speed: %.1f iterations per second" %(len(delta_max_list_3)/(_t2-t1_3)))

show(delta_max_list=delta_max_list_3, phi=phi_3)

In [ ]:
check_answers(phi_3=phi_3, delta_max_list_3=delta_max_list_3, key="answer_13_3a")

By now you probably trust that the technique works, but you can also take a look at $\phi$ just to check if you want:

In [ ]:
# Do not edit or remove the boiler-plate code below.
fig_3 = None
if given(phi_3):

    # Notebook code 
    fig_3 = visualise(phi_3, normalisation=0.5, line_cut_y=80)
    fig_3.show()

show(phi_3=phi_3)

### Exercise 10.6

Make a plot of `delta_max` vs iteration number for the three techniques.

In [ ]:
# Your code here:

# Do not edit or remove the boiler-plate code below.
show()

Note that SOR initially is slower than the other two techniques (and sometimes the convergence parameter can even INCREASE at the start with SOR), but it quickly overtakes the other two and converges very quickly to the correct answer. 

## Including fixed charges

In some cases, one might want to simulate the case where there are external fixed charges in our simulation.

To do this, we need to add an additional term to our update equations to account for the external charge.
In the Jacobi method, we would have:

$$
\phi_{N+1}(x,y) = 
\frac{\phi_N(x+a,y) + 
\phi_N(x-a,y) + 
\phi_N(x,y+a) + 
\phi_N(x,y-a) + \rho(x,y)}{4}
$$

Note that we will work in "dimensionless units", in which we have chosen $\epsilon = 1$ and a lattice spacing $a = 1$ (otherwise the charge term in the update above would be $a^2\rho$).
This will make it easier to code the problem.
You might ask, but what if we want to map the solutions we find to a real charge distribution?
Fortunately, the Poisson equation is linear, and so we can always afterwards rescale our calculated results with the appropriately chosen factors to be able to calculate charges in Coulombs and electrostatic potentials in Volts.
But we will keep it simple and do the calculation in dimensionless units. 

Note that now that externally imposed charge $\rho$ is not zero, one needs to now account for the lattice spacing correctly to calculate the correct potential to achieve the correct values of the source charges. 

For implementing SOR, we would then follow the Gauss-Seidel approach for the updating of $\phi$ and also over-relax.

In our simulation, we will also now need an additional matrix `rho` to specify the positions of fixed charge densities in the simulation.

### Exercise 10.7

Perform a simulation of a 2D "point source" consisting of a single pixel at position `i,j = 50,50` with charge `rho = 1`.
(Note that our "point source" 2D simulation actually corresponds to the voltage around a thin, infinitely long wire.) Use the metallic boundary condition V=0 at the boundaries. 

To match the numerical answer values of the reference solution, use SOR with an SOR parameter of 1.95.

In [ ]:
# Fill these in as you work through the exercise.
phi_4 = None

target_accuracy_4 = 1e-4
M_3 = 101
SOR_parameter_2 = 1.95

# Our fixed charge input matrix
rho_fixed = np.zeros([M_3,M_3])
rho_fixed[M_3//2,M_3//2] = 1

# phi_4 = ...

# Do not edit or remove the boiler-plate code below.
show(phi=phi_4)

In [ ]:
check_answers(phi_4=phi_4, rho_fixed=rho_fixed, key="answer_13_4a", when=given(phi_4))

In [ ]:
# Do not edit or remove the boiler-plate code below.
fig_4 = None
if given(phi_4):

    # Notebook code
    fig_4 = visualise(phi_4, normalisation=0.5, line_cut_y=80)
    fig_4.show()

show(phi_4=phi_4)

## Calculating charge distributions

In all of the above simulations, even ones with external charge $\rho=0$, there was charge present in the resulting calculated physical problem.
How do we know this?
If there is no charge, there will be no change in potential.
OK, but where is this charge? 

With the exception of the last simulation, the charge in the simulation was _induced_ charge that formed due to the voltage applied to the metal boundaries. 

Even in the last simulation of Exercise 10.7, where there is very clearly charge in the middle of the simulation where we placed the fixed  charge, there is also charge induced on the boundary walls: the image charge induced by the electrical fields from the point charge we added in the middle. 

How do we calculate this charge?
Well, if we know $\phi(x,y)$, we can just use the Poisson equation:

$$
\rho(x,y) = -\nabla^2 \phi(x,y)
$$

where numerically, we use the following to calculate $\nabla^2 \phi$:

$$
\frac{\partial^2\phi}{\partial x^2} +
\frac{\partial^2\phi}{\partial y^2} \approx 
\frac{
\phi(x+a,y) + 
\phi(x-a,y) + 
\phi(x,y+a) + 
\phi(x,y-a) -
4 \phi(x,y)}
{a^2}
$$

A subtle point arises at the boundary: the outer boundary points don't have neighbors on all sides.
For example, a point on the left of the simulation has no point at $x-a$. 

What do we do?
One could add special code for calculating the charge for these boundary conditions, but this would be a lot of cumbersome work. 

A very simple and pragmatic way around this for the "metallic" boundary condition is just to add an extra grid point: make the boundaries two grid points thick.
Since the induced charge in a (non-quantum-mechanical) metal is a perfect surface layer, we know that these "extra" boundary points must have zero charge.
We can then just set them all to zero, and then use the Poisson equation above to calculate the charge density everywhere, just skipping the outermost pixels. 

Let's implement this for the problem of Exercise 10.7.

### Exercise 10.8

Implement Exercise 10.7 with an extra boundary grid point on all sides.

In [ ]:
target_accuracy_5 = 1e-4
M_4 = 103

phi_5 = np.zeros([M_4,M_4])
delta_2 = phi_5.copy()

# A matrix to keep track of which points in our simulation 
# are fixed voltages. In this case, it is only the boundaries
# of the simulation
fixed_3 = np.zeros([M_4,M_4])
fixed_3[:,:] = False

# fixed_3[[___,___,___,___],:] = True
# fixed_3[:,[___,___,___,___]] = True

# The external fixed charge
rho_fixed_2 = np.zeros([M_4,M_4])

# rho_fixed_2[___,___] = 1

# Set `unfinished_3` to False once you have written the code below, so that it can run.
unfinished_3 = True

# Do not edit or remove the boiler-plate code below.
show()

In [ ]:
check_answers(rho_fixed_2=rho_fixed_2, phi_5=phi_5, key="answer_13_5a", when=not unfinished_3)

Now that we have the calculated potential with the "extra boundary" point, we can calculate the charge density in our simulation, including the charge density induced on the boundary walls, by using the equation below:

$$
\rho(x,y) = 
-\Big( 
\phi(x+a,y) + 
\phi(x-a,y) + 
\phi(x,y+a) + 
\phi(x,y-a) -
4 \phi(x,y)
\Big)
$$

Note that you can only apply this formula to the "interior" points of the simulation: for the "boundary points" at the edges of the simulation, you have no neighbors to fill in this equation!
This leads to a problem: the matrix we will calculate for the charge density will be two pixels smaller in each direction (an MxM matrix for $\phi$ becomes an (M-2)x(M-2) matrix for the calculated $\rho$). 

However, in the example above, we have been using a "double pixel" metal (fixed voltage, [Dirichlet](https://en.wikipedia.org/wiki/Dirichlet_boundary_condition)) boundary condition.
In this case, we know that any electric fields from charges inside the simulation must be fully screened by the first metal pixel, and cannot induce any charge in the second layer of "metal" pixels behind them.
Because of this, if we want to (and it is handy to, for example, for plotting), we can make the calculated $\rho$ matrix also MxM pixels and just set the outermost ones to zero.
And if we create the matrix using `np.zeros` as below, then they already are zero, which is handy!

### Exercise 10.9

Calculate the (dimensionless) charge density for the simulation of Exercise 10.8.

In [ ]:
# The potential you found in the previous exercise is in `phi_5`.
rho_calc = np.zeros([M_4,M_4])

# Do not edit or remove the boiler-plate code below.
show(rho_calc=rho_calc)

In [ ]:
check_answers(rho_calc=rho_calc, key="answer_13_5b", when=not unfinished_3)

You can use this code to make a plot of the calculated surface charge density.
We have chosen a range of the colorscale such that it is not overwhelmed by the large charge density of the point source.

In [ ]:
# Do not edit or remove the boiler-plate code below.
_ax = None
if given(rho_calc):
    _fig, _ax = plt.subplots()
    _im1 = _ax.imshow(rho_calc, cmap="RdBu")
    _fig.colorbar(_im1)
    _im1.set_clim(vmax=-np.min(rho_calc)-2e-6)

show(_ax, rho_calc=rho_calc)

You can also "explore" the image here and look at some line cuts.
The plot below uses a strongly non-linear colour scale (`normalisation=0.148`), which brings out the small image charges on the edges next to the large point charge, and an autoscaled line cut along row 1 (`line_cut_y=1`).

What strange thing do you see if you move the line cut to `line_cut_y=2`?
If you look at the linecut, you should see that something is wrong.
Can you spot it?
And do you know where it comes from?
And how to solve it? 

In [ ]:
# Do not edit or remove the boiler-plate code below.
fig_5 = None
if given(rho_calc):
    fig_5 = visualise(rho_calc, normalisation=0.148, line_cut_y=1, autoscale=True)
    fig_5.show()

show(rho_calc=rho_calc)

What do you expect the total charge to be in your simulation?
Use `np.sum()` to check your answer.
Compare this number to the external charge you put on the center pixel in the simulation.
Does this make sense?
How could you make it better? 

In [ ]:
# Do not edit or remove the boiler-plate code below.
if given(rho_calc):
    np.sum(rho_calc)

show(rho_calc=rho_calc)